# Ensemble Clássico RF + XGBoost

Combina os dois modelos com melhor desempenho no estudo (Random Forest e XGBoost) por média das réplicas: as predições bootstrap dos dois são reunidas, e a mediana e os intervalos resultam do conjunto combinado. A ideia é reduzir a variância aproveitando os dois melhores classificadores clássicos. Kernel: `qml_dengue`.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, json, sys, time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(REPO_ROOT, "src")); sys.path.insert(0, os.path.abspath(".."))
from feature_engineering import construir_features, splits_validacao
from utils_qml import metricas, salvar_padrao, plot_pred, validar_json_saida
CACHE = os.path.join(REPO_ROOT, "data", "dados_dengue_df_real.json")
dados_brutos = json.load(open(CACHE, encoding="utf-8"))
dataset = construir_features(dados_brutos, n_lags=4); splits = splits_validacao(dataset)
NOMES = {0:"C1",1:"C2",2:"C3"}; CENARIOS={}
for i,sp in enumerate(splits[:3]):
    tr,te=sp["treino"],sp["teste"]
    CENARIOS[NOMES[i]]={"X_train":np.array(tr["X"]),"y_train":np.array(tr["y"]),
                        "X_test":np.array(te["X"]),"y_test":np.array(te["y"])}
print("Cenários:", {c:(len(d["X_train"]),len(d["X_test"])) for c,d in CENARIOS.items()})

Cenários: {'C1': (36, 143), 'C2': (88, 91), 'C3': (125, 54)}


In [2]:
# ── Ensemble: 5 réplicas RF + 5 réplicas XGBoost combinadas ──
N_BOOT = 5
np.random.seed(42); RESULTADOS = {}
for cen, d in CENARIOS.items():
    t0=time.time()
    Xtr,ytr,Xte,yte = d["X_train"],d["y_train"],d["X_test"],d["y_test"]
    n=len(Xtr); w=1.0+2.0*(np.arange(n)/max(n-1,1))
    preds=np.zeros((2*N_BOOT,len(Xte)))
    for b in range(N_BOOT):
        rng=np.random.RandomState(42+b); idx=rng.choice(n,n,replace=True)
        rf=RandomForestRegressor(n_estimators=200,max_depth=12,min_samples_leaf=3,random_state=42+b,n_jobs=-1)
        rf.fit(Xtr[idx],np.log1p(ytr[idx])); preds[b]=np.maximum(np.expm1(rf.predict(Xte)),0)
        xg=XGBRegressor(n_estimators=300,max_depth=4,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,
                        min_child_weight=3,reg_lambda=1.0,random_state=42+b,n_jobs=-1)
        xg.fit(Xtr[idx],np.log1p(ytr[idx]),sample_weight=w[idx]); preds[N_BOOT+b]=np.maximum(np.expm1(xg.predict(Xte)),0)
    med=np.median(preds,axis=0); m=metricas(yte,med,preds,nome=f"Ensemble_RFXGB_{cen}")
    RESULTADOS[cen]={**m,"preds_matrix":preds,"mediana":med,"y_test":yte,"tempo_s":time.time()-t0}
    print(f"{cen}: R2={m['R2']:.4f} | WIS={m['WIS']:.2f} | {RESULTADOS[cen]['tempo_s']:.0f}s")

C1: R2=0.2230 | WIS=1670.56 | 1s
C2: R2=0.1342 | WIS=2576.44 | 1s
C3: R2=0.1436 | WIS=117.47 | 1s


In [3]:
print("="*60)
print("ENSEMBLE CLÁSSICO RF + XGBoost".center(60))
print("="*60)
for cen, r in RESULTADOS.items():
    print(f"{cen:<8} R2={r['R2']:.4f}  WIS={r['WIS']:.2f}")
print("="*60)
plot_pred(RESULTADOS, "Ensemble RF+XGBoost: Predição vs. Observado (DF 2022-2025)", "ensemble_rfxgb_pred_vs_obs.png")


               ENSEMBLE CLÁSSICO RF + XGBoost               
C1       R2=0.2230  WIS=1670.56
C2       R2=0.1342  WIS=2576.44
C3       R2=0.1436  WIS=117.47
[SALVO] ensemble_rfxgb_pred_vs_obs.png


In [4]:
SCHEMA_INFO={"algoritmo":"Ensemble_RF_XGBoost","fase":42,"tipo":"ensemble_classico",
             "n_parametros_quanticos":None,"config":{"modelos":["RandomForest","XGBoost"],"n_bootstrap_cada":N_BOOT,"combinacao":"mediana das reunidas"}}
doc=salvar_padrao(RESULTADOS,SCHEMA_INFO); validar_json_saida(doc,contexto="Ensemble_RF_XGBoost")

[PADRAO] fase42_ensemble_rf_xgboost_resultados.json
  Algoritmo : Ensemble_RF_XGBoost
  Tipo      : ensemble_classico
  Parametros quanticos: None
  C1: R2=+0.2230 | WIS=1670.56 | 1.4s
  C2: R2=+0.1342 | WIS=2576.44 | 1.3s
  C3: R2=+0.1436 | WIS=117.47 | 1.3s
[CONTRATO OK] [Ensemble_RF_XGBoost] JSON valido — todos os campos e invariantes corretos


True